<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_08_practicum_big_o/note_lesson_08_big_o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 8 — Практикум П1. Big O + базові задачі

> Перший практикум алгоритмічної трійки М1 (П1 → П2 → П3):
>
> ```
>               ALGORITHMIC THINKING
>                      │
>           ┌──────────┼──────────┐
>           ▼          ▼          ▼
>        П1            П2          П3
>     EVALUATE        CHOOSE     REPRESENT
>    complexity      strategy    data structure
> ```
>
> **Це не урок про перевірку правильності відповіді.** Два рішення можуть обидва бути правильними — і все одно коштувати по-різному. Тут з'являється нова вісь мислення: **коректність ≠ ефективність**. Матеріал навмисно обмежений `O(1)` / `O(n)` / `O(n²)` — без формального аналізу, без `O(log n)`/`O(n log n)` (це буде в Практикумі П2) і без внутрішньої механіки CPython.

Структура: **RETRIEVE → CONCEPT → PREDICT → RUN/INVESTIGATE → CREATE → TRANSFER**.

## 🔁 RETRIEVE — пригадай попередні уроки (без підглядання)

1. Що робить `for x in range(n): ...` — скільки разів виконається тіло циклу?
2. Що станеться, якщо один `for`-цикл покласти всередину тіла іншого `for`-циклу?
3. Як перевірити, чи є значення `x` у списку `data` — яким виразом?

<details>
<summary>Відповіді</summary>

1. Рівно `n` разів — по одному на кожне число від `0` до `n - 1`.
2. Внутрішній цикл повністю виконується заново на **кожній** ітерації зовнішнього — якщо зовнішній робить `n` кроків, а внутрішній теж `n`, разом це `n × n = n²` виконань тіла.
3. `x in data`.

</details>

## 📖 CONCEPT — два правильні рішення, різна ціна

Задача: чи є в списку повторювані елементи? Два способи — обидва дають правильну відповідь:

In [1]:
def has_duplicates_v1(data):
    """Перевіряє кожну пару елементів."""
    for i in range(len(data)):
        for j in range(i + 1, len(data)):
            if data[i] == data[j]:
                return True
    return False


def has_duplicates_v2(data):
    """Запам'ятовує побачене в set."""
    seen = set()
    for value in data:
        if value in seen:
            return True
        seen.add(value)
    return False


sample = [3, 1, 4, 1, 5]
print(has_duplicates_v1(sample))   # True
print(has_duplicates_v2(sample))   # True

assert has_duplicates_v1(sample) == has_duplicates_v2(sample) == True
assert has_duplicates_v1([1, 2, 3]) == has_duplicates_v2([1, 2, 3]) == False
print("OK — обидва рішення дають однакову, правильну відповідь")

True
True
OK — обидва рішення дають однакову, правильну відповідь


**Обидва рішення правильні.** Але `has_duplicates_v1` для кожного елемента заново переглядає (майже) весь список — вкладений цикл, тіло виконується приблизно `n × n` разів. `has_duplicates_v2` дивиться на кожен елемент рівно один раз і використовує `set` для миттєвої перевірки «чи бачили ми це вже». Яке рішення дорожче — і наскільки? Щоб відповісти чесно, не вгадуючи, будемо **рахувати операції**, а не покладатись на відчуття «здається, швидше».

### Три рівні, які нам знадобляться сьогодні

| Позначення | Що означає | Приклад |
|---|---|---|
| `O(1)` | стала кількість дій, не залежить від розміру даних | `data[0]`, `len(data)` |
| `O(n)` | кількість дій росте **пропорційно** розміру даних | один прохід циклом по `n` елементах |
| `O(n²)` | кількість дій росте пропорційно **квадрату** розміру даних | цикл усередині циклу, обидва по `n` елементах |

Це не формальне означення — рівно стільки, скільки потрібно для наступного кроку.

## 🔮 PREDICT

Список `sample` має 5 елементів. Уяви список у **10 разів більший** (50 елементів, без дублікатів — найгірший випадок, обидві функції мусять дійти до кінця). У скільки разів більше операцій зробить `has_duplicates_v1`? А `has_duplicates_v2`? Запиши прогноз (наприклад: «v1 — приблизно в X разів більше, v2 — приблизно в Y разів більше»), перш ніж дивитись на код нижче.

## ▶️ RUN / INVESTIGATE

Рахуємо реальну кількість порівнянь (не час виконання — час залежить від конкретного комп'ютера, а кількість операцій — ні):

In [2]:
def has_duplicates_v1_counted(data):
    comparisons = 0
    for i in range(len(data)):
        for j in range(i + 1, len(data)):
            comparisons += 1
            if data[i] == data[j]:
                return True, comparisons
    return False, comparisons


def has_duplicates_v2_counted(data):
    comparisons = 0
    seen = set()
    for value in data:
        comparisons += 1
        if value in seen:
            return True, comparisons
        seen.add(value)
    return False, comparisons


small = list(range(5))     # 5 елементів, без дублікатів — найгірший випадок
large = list(range(50))    # у 10 разів більше елементів

_, v1_small = has_duplicates_v1_counted(small)
_, v1_large = has_duplicates_v1_counted(large)
_, v2_small = has_duplicates_v2_counted(small)
_, v2_large = has_duplicates_v2_counted(large)

print(f"v1 (вкладені цикли): n=5 -> {v1_small} порівнянь, n=50 -> {v1_large} порівнянь (x{v1_large / v1_small:.1f})")
print(f"v2 (set):             n=5 -> {v2_small} порівнянь, n=50 -> {v2_large} порівнянь (x{v2_large / v2_small:.1f})")

v1 (вкладені цикли): n=5 -> 10 порівнянь, n=50 -> 1225 порівнянь (x122.5)
v2 (set):             n=5 -> 5 порівнянь, n=50 -> 50 порівнянь (x10.0)


**Порівняй зі своїм прогнозом.** `v2` виконує рівно в 10 разів більше операцій при вході в 10 разів більшому — це і є `O(n)`: пропорційно розміру даних. `v1` виконує приблизно в **100 разів** більше — `10²`, це і є `O(n²)`: подвійний вкладений цикл, ціна росте як квадрат розміру.

In [3]:
assert abs(v2_large / v2_small - 10) < 0.01                 # O(n): рівно пропорційно
assert v1_large / v1_small > 80                              # O(n²): близько до 100, не до 10
print(f"OK — v2 росте лінійно (x{v2_large / v2_small:.1f}), v1 росте квадратично (x{v1_large / v1_small:.1f})")

OK — v2 росте лінійно (x10.0), v1 росте квадратично (x122.5)


## 🛠️ CREATE — три задачі

Для кожної задачі: напиши розв'язок, потім сам визнач його клас складності (`O(1)` / `O(n)` / `O(n²)`) — обґрунтування одним реченням: скільки разів виконається основна дія відносно розміру входу.

### Задача 1 — FizzBuzz

Для чисел від `1` до `n` включно: кратне 3 і 5 → `"FizzBuzz"`, кратне лише 3 → `"Fizz"`, кратне лише 5 → `"Buzz"`, інакше — саме число (рядком).

In [4]:
def fizzbuzz(n):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    result = []
    for i in range(1, n + 1):
        if i % 15 == 0:
            result.append("FizzBuzz")
        elif i % 3 == 0:
            result.append("Fizz")
        elif i % 5 == 0:
            result.append("Buzz")
        else:
            result.append(str(i))
    return result
    # END SOLUTION


print(fizzbuzz(15))
assert fizzbuzz(5) == ["1", "2", "Fizz", "4", "Buzz"]
assert fizzbuzz(15)[14] == "FizzBuzz"
print("OK — FizzBuzz: один прохід по n чисел -> O(n)")

['1', '2', 'Fizz', '4', 'Buzz', 'Fizz', '7', '8', 'Fizz', 'Buzz', '11', 'Fizz', '13', '14', 'FizzBuzz']
OK — FizzBuzz: один прохід по n чисел -> O(n)


### Задача 2 — паліндром, двома способами

Перевір, чи рядок читається однаково зліва направо і справа наліво. Спочатку — простий спосіб (розвернути й порівняти):

In [5]:
def is_palindrome_v1(text):
    return text == text[::-1]


print(is_palindrome_v1("потоп"))
print(is_palindrome_v1("python"))

assert is_palindrome_v1("потоп") is True
assert is_palindrome_v1("python") is False
assert is_palindrome_v1("") is True          # порожній рядок — паліндром
assert is_palindrome_v1("a") is True          # один символ — завжди паліндром

True
False


`text[::-1]` створює **новий, повністю розвернутий рядок** — навіть якщо перші символи вже не збігаються, робота триває до кінця. Другий спосіб — два вказівники назустріч один одному, з негайним виходом при першій незбіжності:

In [6]:
def is_palindrome_v2(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    left, right = 0, len(text) - 1
    while left < right:
        if text[left] != text[right]:
            return False
        left += 1
        right -= 1
    return True
    # END SOLUTION


assert is_palindrome_v2("потоп") is True
assert is_palindrome_v2("python") is False
assert is_palindrome_v2("") is True
assert is_palindrome_v2("a") is True

for word in ["потоп", "python", "", "a", "abba", "abca"]:
    assert is_palindrome_v1(word) == is_palindrome_v2(word)
print("OK — обидва способи завжди узгоджені; v2 може завершитись раніше, не дочитуючи рядок")

OK — обидва способи завжди узгоджені; v2 може завершитись раніше, не дочитуючи рядок


### Задача 3 — шифр Цезаря

На Уроці 3 був лише невеликий приклад роботи з `ord`/`chr` — готового шифру там не було. Тепер — повна задача: зсунути кожну літеру рядка на `shift` позицій в алфавіті (тільки латинські малі літери `a`-`z`, інші символи лишаються без змін; зсув циклічний — після `z` знову `a`).

In [7]:
def caesar_encode(text, shift):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    result = []
    for char in text:
        if "a" <= char <= "z":
            shifted = (ord(char) - ord("a") + shift) % 26
            result.append(chr(ord("a") + shifted))
        else:
            result.append(char)
    return "".join(result)
    # END SOLUTION


def caesar_decode(text, shift):
    return caesar_encode(text, -shift)


encoded = caesar_encode("hello, world!", 3)
print(encoded)
print(caesar_decode(encoded, 3))

assert caesar_encode("abc", 1) == "bcd"
assert caesar_encode("xyz", 3) == "abc"          # циклічний перехід через кінець алфавіту
assert caesar_decode(caesar_encode("hello, world!", 5), 5) == "hello, world!"
assert caesar_encode("123!", 7) == "123!"        # не-літери без змін
print("OK — шифр Цезаря: один прохід по символах -> O(n)")

khoor, zruog!
hello, world!
OK — шифр Цезаря: один прохід по символах -> O(n)


## 🔄 TRANSFER — порівняй рішення

Ось два рішення задачі «чи всі елементи списку унікальні»:

```python
def all_unique_a(data):
    for i in range(len(data)):
        for j in range(len(data)):
            if i != j and data[i] == data[j]:
                return False
    return True

def all_unique_b(data):
    return len(data) == len(set(data))
```

Визнач клас складності кожного (`O(1)`/`O(n)`/`O(n²)`) і обґрунтуй одним реченням — скільки разів виконається порівняння відносно розміру `data`. Перевір прогноз так само, як у CONCEPT/RUN вище: інструментуй обидві функції лічильником операцій і поріняй ріст на `n` і `10n`.

In [8]:
def all_unique_a_counted(data):
    comparisons = 0
    for i in range(len(data)):
        for j in range(len(data)):
            comparisons += 1
            if i != j and data[i] == data[j]:
                return False, comparisons
    return True, comparisons


def all_unique_b_counted(data):
    # довжина -> set() виконує ~n операцій вставки; рахуємо їх явно замість того, щоб покладатись на вбудовану set()
    comparisons = 0
    seen = set()
    for value in data:
        comparisons += 1
        seen.add(value)
    return len(seen) == len(data), comparisons


# YOUR CODE HERE — визнач класи складності та перевір інструментуванням
# BEGIN SOLUTION
small = list(range(6))
large = list(range(60))

_, a_small = all_unique_a_counted(small)
_, a_large = all_unique_a_counted(large)
_, b_small = all_unique_b_counted(small)
_, b_large = all_unique_b_counted(large)

ratio_a = a_large / a_small
ratio_b = b_large / b_small

print(f"all_unique_a: x{ratio_a:.1f} при 10x вході  -> O(n²)")
print(f"all_unique_b: x{ratio_b:.1f} при 10x вході  -> O(n)")

assert ratio_a > 80          # близько до 100 -> O(n²)
assert abs(ratio_b - 10) < 0.01   # рівно 10 -> O(n)
# END SOLUTION
print("OK — all_unique_a: O(n²) (подвійний цикл по всіх парах), all_unique_b: O(n) (один прохід + set)")

all_unique_a: x100.0 при 10x вході  -> O(n²)
all_unique_b: x10.0 при 10x вході  -> O(n)
OK — all_unique_a: O(n²) (подвійний цикл по всіх парах), all_unique_b: O(n) (один прохід + set)


## ✅ Самоперевірка (5 запитань)

**1.** Функція має один цикл `for x in data: ...` без вкладених циклів усередині. Який це клас складності?

<details><summary>Відповідь</summary><code>O(n)</code> — тіло циклу виконується рівно стільки разів, скільки елементів у <code>data</code>, пропорційно розміру входу.</details>

**2.** `has_duplicates_v1` і `has_duplicates_v2` (з CONCEPT) дають однакову відповідь на будь-якому вході. Чи означає це, що вони однаково ефективні?

<details><summary>Відповідь</summary>Ні. Коректність (обидва дають правильну відповідь) і ефективність (скільки операцій знадобилось) — дві незалежні осі. <code>v1</code> — <code>O(n²)</code>, <code>v2</code> — <code>O(n)</code>, хоча відповідь завжди та сама.</details>

**3.** Чому для порівняння двох рішень краще рахувати кількість операцій, а не вимірювати час виконання секундоміром?

<details><summary>Відповідь</summary>Час виконання залежить від конкретного комп'ютера, навантаження системи в момент запуску тощо — його важко відтворити. Кількість операцій — властивість самого алгоритму, однакова на будь-якій машині, тому це чесніший спосіб порівняння.</details>

**4.** `is_palindrome_v2` (два вказівники) інколи виконує менше порівнянь, ніж `is_palindrome_v1` (розвернути й порівняти). Коли саме?

<details><summary>Відповідь</summary>Коли перша ж пара символів (з країв) не збігається — <code>v2</code> одразу повертає <code>False</code>, не перевіряючи решту рядка. <code>v1</code> завжди спершу будує повністю розвернуту копію рядка, незалежно від того, де саме різниця.</details>

**5.** У `all_unique_a` (TRANSFER) внутрішній цикл також іде по `range(len(data))`, а не `range(i + 1, len(data))`, як у `has_duplicates_v1`. Чи змінює це клас складності?

<details><summary>Відповідь</summary>Ні. <code>range(i + 1, len(data))</code> і <code>range(len(data))</code> відрізняються приблизно вдвічі за кількістю операцій (трикутник пар проти повного квадрата), але обидва все одно ростуть пропорційно <code>n²</code> — сталий множник (тут ×2) не змінює клас складності.</details>

### Шпаргалка

| Клас | Ознака в коді | Приклад із уроку |
|---|---|---|
| `O(1)` | немає циклу по розміру даних | `data[0]`, `len(data)` |
| `O(n)` | один прохід циклом по `n` елементах | `has_duplicates_v2`, FizzBuzz, шифр Цезаря |
| `O(n²)` | цикл усередині циклу, обидва по `n` | `has_duplicates_v1`, `all_unique_a` |

**Правило перевірки прогнозу:** інструментуй код лічильником операцій, порівняй вхід `n` і `10n` — приблизно ×10 означає `O(n)`, приблизно ×100 означає `O(n²)`.

## Далі

Практикум П2 (позиція 11, «Пошук») бере цю саму вісь мислення далі: «Можу просто перебрати все» → «А чи знаю я щось про дані, що дозволить зробити краще?» — лінійний пошук, бінарний пошук, два вказівники, sliding window.